# Lab 12. Spectral Analysis and the Frequency Domain

[Open this notebook in Google Colab](https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-12-spectral-analysis-frequency-domain-lab.ipynb)

This lab is designed for independent study. You do not need to have seen the code before. The notebook first explains the mathematical ideas, then asks you to run small Python experiments, interpret the plots, and answer short questions.

## Learning goals

By the end of this lab, you should be able to:

1. Explain the difference between the time-domain and frequency-domain views of a time series.
2. Interpret frequency, period, amplitude, and phase in a sinusoidal signal.
3. Demonstrate aliasing for discretely observed time series.
4. Simulate random sinusoidal processes and connect their autocovariance to cosine functions.
5. Explain why white noise has a flat spectrum.
6. Compare the theoretical spectra of AR(1) and MA(1) processes.
7. Use the spectrum to describe whether a process is dominated by low-frequency or high-frequency variation.

## How to use this lab

Read each short background section before running the code. After each plot, pause and write a one- or two-sentence interpretation. The goal is not only to produce figures, but to learn what the figures mean.

## 1. Background: time domain versus frequency domain

A time series can be studied in two complementary ways.

The **time-domain view** asks how the present value depends on the past. For example, an AR(1) model has the form

$$
X_t = \phi X_{t-1} + W_t.
$$

This model directly describes how the series moves forward in time.

The **frequency-domain view** asks which oscillations explain the variation in the series. Instead of asking whether $X_t$ depends on $X_{t-1}$, we ask whether the series contains slow waves, fast waves, or a mixture of waves.

A basic sinusoid has the form

$$
y_t = A \cos(2\pi \omega t + \varphi),
$$

where:

- $A$ is the amplitude.
- $\omega$ is the frequency, measured in cycles per time unit.
- $1/\omega$ is the period, measured in time units per cycle.
- $\varphi$ is the phase shift.

A low frequency means a slowly moving wave. A high frequency means a rapidly oscillating wave.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(7339)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True


def plot_series(x, title="Time series", xlabel="time", ylabel="value"):
    # Simple helper for time-series plots.
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(np.arange(len(x)), x, lw=2)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    plt.show()


def sample_acf(x, max_lag):
    # Biased sample autocorrelation, useful for visualization.
    x = np.asarray(x, dtype=float)
    x = x - x.mean()
    n = len(x)
    gamma0 = np.mean(x * x)
    values = []
    for h in range(max_lag + 1):
        values.append(np.mean(x[: n - h] * x[h:]) / gamma0)
    return np.array(values)


def plot_acf_values(acf_values, title="Sample ACF"):
    fig, ax = plt.subplots(figsize=(10, 4))
    lags = np.arange(len(acf_values))
    ax.axhline(0, color="black", lw=1)
    ax.vlines(lags, 0, acf_values, lw=2)
    ax.plot(lags, acf_values, "o")
    ax.set_title(title)
    ax.set_xlabel("lag")
    ax.set_ylabel("autocorrelation")
    plt.show()


def periodogram(x):
    # Return one-sided Fourier frequencies and a simple periodogram.
    x = np.asarray(x, dtype=float)
    x = x - x.mean()
    n = len(x)
    freqs = np.fft.rfftfreq(n, d=1.0)
    fft_vals = np.fft.rfft(x)
    power = (np.abs(fft_vals) ** 2) / n
    return freqs, power


def plot_periodogram(x, title="Periodogram", max_freq=0.5):
    freqs, power = periodogram(x)
    fig, ax = plt.subplots(figsize=(10, 4))
    mask = freqs <= max_freq
    ax.plot(freqs[mask], power[mask], lw=2)
    ax.set_title(title)
    ax.set_xlabel("frequency in cycles per time unit")
    ax.set_ylabel("periodogram power")
    plt.show()

## 2. Sinusoids: frequency, period, amplitude, and phase

The next cell creates several cosine waves. Compare their frequencies and periods.

If $\omega = 1/20$, the period is $20$. This means one full cycle takes 20 time steps.

If $\omega = 1/5$, the period is $5$. This means the wave oscillates much faster.

In [ ]:
n = 120
t = np.arange(n)

freq_slow = 1 / 30
freq_medium = 1 / 12
freq_fast = 1 / 5

slow_wave = np.cos(2 * np.pi * freq_slow * t)
medium_wave = np.cos(2 * np.pi * freq_medium * t)
fast_wave = np.cos(2 * np.pi * freq_fast * t)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(t, slow_wave, label="frequency = 1/30, period = 30", lw=2)
ax.plot(t, medium_wave, label="frequency = 1/12, period = 12", lw=2)
ax.plot(t, fast_wave, label="frequency = 1/5, period = 5", lw=2)
ax.set_title("Cosine waves with different frequencies")
ax.set_xlabel("time")
ax.set_ylabel("value")
ax.legend()
plt.show()

### Interpretation checkpoint

Answer in your own words:

1. Which curve changes most slowly?
2. Which curve oscillates most rapidly?
3. What happens to the period when the frequency increases?

## 3. A signal with multiple frequencies

A real time series may contain several oscillatory components. The next example combines two waves and a small amount of noise:

$$
X_t = 1.5\cos(2\pi t/30) + 0.8\sin(2\pi t/8) + E_t.
$$

The first component has period $30$, and the second has period $8$.

In [ ]:
noise = rng.normal(0, 0.35, size=n)
x_mixed = 1.5 * np.cos(2 * np.pi * t / 30) + 0.8 * np.sin(2 * np.pi * t / 8) + noise

plot_series(x_mixed, title="Signal with periods 30 and 8 plus noise")
plot_periodogram(x_mixed, title="Periodogram of the mixed-frequency signal")

freqs, power = periodogram(x_mixed)
top_idx = np.argsort(power[1:])[-5:][::-1] + 1
pd.DataFrame({
    "frequency": freqs[top_idx],
    "estimated_period": 1 / freqs[top_idx],
    "power": power[top_idx]
})

### Interpretation checkpoint

The true periods are $30$ and $8$. Look at the table above.

- Do the largest periodogram peaks occur near frequencies $1/30$ and $1/8$?
- Why might the estimated periods not be exactly equal to $30$ and $8$?

## 4. Aliasing: different continuous waves can look the same after sampling

In discrete-time data, we only observe values at integer times $t=0,1,2,\ldots$. Because of this, different frequencies may produce the same observed sequence.

For integer $t$,

$$
\cos(2\pi(0.25)t) = \cos(2\pi(0.75)t).
$$

This is an example of **aliasing**. It means a discretely sampled time series cannot always distinguish all continuous-time frequencies.

In [ ]:
t_small = np.arange(16)
x_025 = np.cos(2 * np.pi * 0.25 * t_small)
x_075 = np.cos(2 * np.pi * 0.75 * t_small)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_small, x_025, "o-", label="frequency 0.25")
ax.plot(t_small, x_075, "s--", label="frequency 0.75")
ax.set_title("Aliasing: frequencies 0.25 and 0.75 give identical samples")
ax.set_xlabel("integer time")
ax.set_ylabel("sampled value")
ax.legend()
plt.show()

print("Maximum absolute difference:", np.max(np.abs(x_025 - x_075)))

### Interpretation checkpoint

The maximum absolute difference should be essentially zero. Explain why this makes frequency analysis subtle for discrete-time data.

## 5. Random sinusoidal processes and autocovariance

A deterministic sinusoid has a fixed amplitude and phase. A random sinusoidal process has random coefficients. For example,

$$
X_t = A\cos(2\pi \omega t) + B\sin(2\pi \omega t),
$$

where $A$ and $B$ are independent random variables with mean $0$ and variance $\sigma^2$.

Then:

$$
E[X_t]=0,
$$

and the autocovariance depends only on the lag $h$:

$$
\gamma(h) = \sigma^2 \cos(2\pi\omega h).
$$

This is a key bridge between time-domain dependence and frequency-domain oscillation.

In [ ]:
R = 4000       # number of independent realizations
n = 80         # length of each realization
omega = 1 / 12
sigma = 1.0
t = np.arange(n)

A = rng.normal(0, sigma, size=R)
B = rng.normal(0, sigma, size=R)
X = A[:, None] * np.cos(2 * np.pi * omega * t)[None, :] + B[:, None] * np.sin(2 * np.pi * omega * t)[None, :]

# Empirical covariance across realizations: Cov(X_t, X_{t+h}) averaged over t.
max_lag = 36
emp_gamma = []
for h in range(max_lag + 1):
    covs = []
    for j in range(n - h):
        covs.append(np.mean(X[:, j] * X[:, j + h]))
    emp_gamma.append(np.mean(covs))
emp_gamma = np.array(emp_gamma)

theory_gamma = sigma**2 * np.cos(2 * np.pi * omega * np.arange(max_lag + 1))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(emp_gamma, "o-", label="empirical autocovariance")
ax.plot(theory_gamma, "--", label="theoretical autocovariance")
ax.set_title("Random sinusoid: autocovariance is a cosine in the lag")
ax.set_xlabel("lag h")
ax.set_ylabel("gamma(h)")
ax.legend()
plt.show()

### Interpretation checkpoint

The autocovariance itself oscillates. Explain why this is natural for a process dominated by a single frequency.

## 6. White noise has a flat spectrum

White noise satisfies

$$
\gamma(0)=\sigma^2, \qquad \gamma(h)=0 \text{ for } h\ne 0.
$$

The spectral density is the Fourier transform of the autocovariance sequence. Since white noise has no special lag dependence, no frequency is preferred. Therefore, its spectrum is flat.

In practice, a sample periodogram of white noise is not perfectly flat. It is noisy. But it should not show persistent dominant peaks.

In [ ]:
n = 512
white_noise = rng.normal(0, 1, size=n)

plot_series(white_noise[:160], title="First 160 observations of white noise")
plot_acf_values(sample_acf(white_noise, 40), title="Sample ACF of white noise")
plot_periodogram(white_noise, title="Periodogram of white noise")

### Interpretation checkpoint

White noise can have some large periodogram values by chance. Why should we avoid declaring every isolated periodogram spike to be a real periodic signal?

## 7. Theoretical spectrum of an AR(1) process

For the AR(1) process

$$
X_t = \phi X_{t-1} + W_t,
$$

with $|\phi|<1$ and $W_t$ white noise with variance $\sigma^2$, the spectral density is proportional to

$$
f(\omega) = \frac{\sigma^2}{1 - 2\phi\cos(2\pi\omega)+\phi^2},
\qquad -1/2 \le \omega \le 1/2.
$$

When $\phi>0$, the process is persistent and smooth, so the spectrum is large near frequency $0$.

When $\phi<0$, the process tends to alternate signs, so the spectrum is larger near high frequencies.

In [ ]:
def simulate_ar1(phi, n=600, sigma=1.0, burnin=200):
    total = n + burnin
    w = rng.normal(0, sigma, size=total)
    x = np.zeros(total)
    for t in range(1, total):
        x[t] = phi * x[t - 1] + w[t]
    return x[burnin:]


def ar1_spectrum(phi, omega, sigma=1.0):
    return sigma**2 / (1 - 2 * phi * np.cos(2 * np.pi * omega) + phi**2)

omega_grid = np.linspace(0, 0.5, 400)

x_pos = simulate_ar1(phi=0.85)
x_neg = simulate_ar1(phi=-0.85)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes[0, 0].plot(x_pos[:160])
axes[0, 0].set_title("AR(1), phi = 0.85: smooth/persistent")
axes[0, 1].plot(x_neg[:160])
axes[0, 1].set_title("AR(1), phi = -0.85: alternating/rough")
axes[1, 0].plot(omega_grid, ar1_spectrum(0.85, omega_grid))
axes[1, 0].set_title("Theoretical spectrum, phi = 0.85")
axes[1, 0].set_xlabel("frequency")
axes[1, 1].plot(omega_grid, ar1_spectrum(-0.85, omega_grid))
axes[1, 1].set_title("Theoretical spectrum, phi = -0.85")
axes[1, 1].set_xlabel("frequency")
plt.tight_layout()
plt.show()

### Interpretation checkpoint

1. For $\phi=0.85$, where is the spectrum largest?
2. For $\phi=-0.85$, where is the spectrum largest?
3. How does this match the time plots?

## 8. Theoretical spectrum of an MA(1) process

For the MA(1) process

$$
X_t = W_t + \theta W_{t-1},
$$

with white-noise variance $\sigma^2$, the spectral density is proportional to

$$
f(\omega) = \sigma^2\left(1+\theta^2+2\theta\cos(2\pi\omega)\right).
$$

Positive $\theta$ emphasizes low frequencies. Negative $\theta$ emphasizes high frequencies.

In [ ]:
def simulate_ma1(theta, n=600, sigma=1.0, burnin=200):
    total = n + burnin
    w = rng.normal(0, sigma, size=total + 1)
    x = np.zeros(total)
    for t in range(total):
        x[t] = w[t + 1] + theta * w[t]
    return x[burnin:]


def ma1_spectrum(theta, omega, sigma=1.0):
    return sigma**2 * (1 + theta**2 + 2 * theta * np.cos(2 * np.pi * omega))

x_ma_pos = simulate_ma1(theta=0.9)
x_ma_neg = simulate_ma1(theta=-0.9)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes[0, 0].plot(x_ma_pos[:160])
axes[0, 0].set_title("MA(1), theta = 0.9")
axes[0, 1].plot(x_ma_neg[:160])
axes[0, 1].set_title("MA(1), theta = -0.9")
axes[1, 0].plot(omega_grid, ma1_spectrum(0.9, omega_grid))
axes[1, 0].set_title("Theoretical spectrum, theta = 0.9")
axes[1, 0].set_xlabel("frequency")
axes[1, 1].plot(omega_grid, ma1_spectrum(-0.9, omega_grid))
axes[1, 1].set_title("Theoretical spectrum, theta = -0.9")
axes[1, 1].set_xlabel("frequency")
plt.tight_layout()
plt.show()

### Interpretation checkpoint

Compare the AR(1) and MA(1) spectra. Which model can produce a sharper low-frequency peak when its parameter is near $1$?

## 9. Variance as total spectral power

A useful interpretation is:

$$
\gamma(0) = Var(X_t) = \int_{-1/2}^{1/2} f(\omega)\,d\omega.
$$

In words: the total area under the spectral density equals the variance of the process.

We can check this numerically for the AR(1) spectrum. Because the one-sided plot from $0$ to $1/2$ contains half the symmetric spectrum except endpoints, we approximate the integral on $[-1/2,1/2]$ directly.

In [ ]:
phi = 0.7
sigma = 1.0
omega_full = np.linspace(-0.5, 0.5, 20001)
f_full = ar1_spectrum(phi, omega_full, sigma=sigma)
area = np.trapz(f_full, omega_full)
theoretical_variance = sigma**2 / (1 - phi**2)

print("Numerical integral of spectral density:", area)
print("Theoretical AR(1) variance:", theoretical_variance)
print("Absolute difference:", abs(area - theoretical_variance))

### Interpretation checkpoint

The numerical integral should be close to the theoretical variance. Explain why this identity supports the interpretation of spectral density as variance distributed across frequencies.

## 10. Guided practice: identify the dominant frequency

The next cell hides the frequency used to generate a signal. Your task is to estimate its period using the periodogram.

In [ ]:
n = 240
t = np.arange(n)
hidden_period = 24
x_hidden = 1.2 * np.cos(2 * np.pi * t / hidden_period + 0.7) + rng.normal(0, 0.6, size=n)

plot_series(x_hidden, title="Hidden-period signal")
plot_periodogram(x_hidden, title="Periodogram of hidden-period signal")

freqs, power = periodogram(x_hidden)
best = np.argmax(power[1:]) + 1
estimated_frequency = freqs[best]
estimated_period = 1 / estimated_frequency
print("Estimated dominant frequency:", estimated_frequency)
print("Estimated period:", estimated_period)

### Interpretation checkpoint

Without looking back at the code, write down the estimated dominant period. Then compare it with the true hidden period shown in the code.

## 11. Mini-project

Choose one of the following mini-projects.

### Option A: AR(1) spectrum comparison

1. Simulate AR(1) processes with $\phi=-0.9,-0.5,0,0.5,0.9$.
2. Plot the first 120 observations of each process.
3. Plot the theoretical spectrum for each process.
4. Explain how the time-domain appearance changes as $\phi$ moves from negative to positive.

### Option B: Mixed-frequency signal

1. Create a time series that combines two periodic components and noise.
2. Compute its periodogram.
3. Identify the two strongest frequencies.
4. Explain which periods are easiest or hardest to detect.

### Option C: Aliasing experiment

1. Choose a frequency $\omega$ between $0$ and $0.5$.
2. Compare the sampled cosine with frequency $1-\omega$.
3. Verify that they match at integer times.
4. Explain why the frequency range $[0,1/2]$ is special for real-valued discrete-time data.

## 12. AI-assisted study prompts

Use an AI assistant as a tutor, not as a replacement for your own reasoning. Good prompts include the data-generating process, what you plotted, and what you think the result means.

Try these prompts:

1. "I simulated an AR(1) process with phi = 0.85. The spectrum is largest near frequency 0. Explain why this means the series is smooth in the time domain."
2. "I simulated an AR(1) process with phi = -0.85. Why does the spectrum become large near frequency 0.5?"
3. "Explain aliasing for discrete-time cosine waves using the example frequencies 0.25 and 0.75."
4. "Give me a checklist for deciding whether a periodogram peak is likely to represent a real periodic component or random noise."
5. "I found a dominant period in a time series. What time-domain plots should I check before trusting this frequency-domain conclusion?"

After using AI, verify every claim by returning to the formulas and plots in this notebook.

## 13. Lab checklist

Before leaving this lab, make sure you can do the following without looking up the answer:

- Define frequency, period, amplitude, and phase.
- Explain the difference between time-domain and frequency-domain analysis.
- Demonstrate aliasing with two sampled cosine waves.
- State why white noise has a flat spectrum.
- Interpret the AR(1) spectrum when $\phi>0$ and when $\phi<0$.
- Interpret the MA(1) spectrum when $\theta>0$ and when $\theta<0$.
- Use a periodogram to estimate a dominant period.
- Explain why a periodogram peak needs careful interpretation.